<a href="https://colab.research.google.com/github/amirhamrouni/Alala/blob/main/docs/MoneyPrinterTurbo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MoneyPrinterTurbo + Arabic RTL + API on Colab
تشغيل نظيف لـ MoneyPrinterTurbo v1.3.7 مع WebUI، API على 8090، ودعم عربي RTL. خزّن `NGROK_AUTHTOKEN` في Colab Secrets بدل وضعه داخل الكود.

In [ ]:
import os, re, subprocess, time, urllib.request
from pathlib import Path
from getpass import getpass

REPO = Path('/content/MoneyPrinterTurbo')
UPSTREAM = 'https://github.com/harry0703/MoneyPrinterTurbo.git'
PATCH_URL = 'https://raw.githubusercontent.com/amirhamrouni/Alala/main/moneyprinter-mobile-gateway/scripts/patch_moneyprinter_arabic.py'
API_PORT = 8090
WEB_PORT = 8501

def run(cmd, **kwargs):
    print('$', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(cmd, check=True, **kwargs)

def healthy(url):
    try:
        with urllib.request.urlopen(url, timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

run(['python', '-m', 'pip', 'install', '-q', 'uv', 'pyngrok'])
if (REPO / '.git').is_dir():
    run(['git', '-C', str(REPO), 'pull', '--ff-only'])
else:
    if REPO.exists():
        run(['rm', '-rf', str(REPO)])
    run(['git', 'clone', '--depth', '1', UPSTREAM, str(REPO)])
run(['uv', 'python', 'install', '3.11'])
run(['uv', 'sync', '--frozen', '--python', '3.11'], cwd=REPO)

patch_file = Path('/content/patch_moneyprinter_arabic.py')
urllib.request.urlretrieve(PATCH_URL, patch_file)
run(['python', str(patch_file), '--repo', str(REPO)])

cfg = REPO / 'config.toml'
if not cfg.exists():
    cfg.write_text((REPO / 'config.example.toml').read_text(encoding='utf-8'), encoding='utf-8')
raw = cfg.read_text(encoding='utf-8')
if re.search(r'(?m)^listen_port\s*=\s*\d+', raw):
    raw = re.sub(r'(?m)^listen_port\s*=\s*\d+', f'listen_port = {API_PORT}', raw, count=1)
else:
    raw = f'listen_port = {API_PORT}\n' + raw
cfg.write_text(raw, encoding='utf-8')

for port in (API_PORT, WEB_PORT):
    subprocess.run(['bash', '-lc', f'fuser -k {port}/tcp >/dev/null 2>&1 || true'], check=False)
time.sleep(1)

api_log = open('/content/mpt-api.log', 'w', encoding='utf-8')
web_log = open('/content/mpt-webui.log', 'w', encoding='utf-8')
api_proc = subprocess.Popen(['uv', 'run', 'python', 'main.py'], cwd=REPO, stdout=api_log, stderr=subprocess.STDOUT)
web_proc = subprocess.Popen([
    'uv','run','streamlit','run','webui/Main.py',
    f'--server.port={WEB_PORT}','--server.address=0.0.0.0',
    '--browser.gatherUsageStats=False','--client.toolbarMode=minimal',
    '--server.showEmailPrompt=False'
], cwd=REPO, stdout=web_log, stderr=subprocess.STDOUT)

for _ in range(60):
    if healthy(f'http://127.0.0.1:{API_PORT}/ping') and healthy(f'http://127.0.0.1:{WEB_PORT}/_stcore/health'):
        break
    time.sleep(2)
else:
    api_log.flush(); web_log.flush()
    print(Path('/content/mpt-api.log').read_text(errors='replace')[-2500:])
    print(Path('/content/mpt-webui.log').read_text(errors='replace')[-2500:])
    raise RuntimeError('MoneyPrinterTurbo did not become healthy')

try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    ngrok_token = None
if not ngrok_token:
    ngrok_token = getpass('NGROK_AUTHTOKEN: ').strip()
if not ngrok_token:
    raise RuntimeError('NGROK_AUTHTOKEN is required')

from pyngrok import ngrok
ngrok.set_auth_token(ngrok_token)
ngrok.kill()
web_tunnel = ngrok.connect(addr=f'http://127.0.0.1:{WEB_PORT}', proto='http', bind_tls=True)
print('WEBUI =', web_tunnel.public_url)
try:
    api_tunnel = ngrok.connect(addr=f'http://127.0.0.1:{API_PORT}', proto='http', bind_tls=True)
    print('API_BASE =', api_tunnel.public_url)
    print('API_DOCS =', api_tunnel.public_url + '/docs')
except Exception as exc:
    print('API tunnel unavailable; local API still works:', exc)
    print('API_LOCAL =', f'http://127.0.0.1:{API_PORT}')

print('Arabic subtitles: DejaVuSans + reshaper + bidi enabled')
print('MoneyPrinterTurbo ready')


## API test
بعد تشغيل الخلية السابقة، MoneyPrinter API يكون على `http://127.0.0.1:8090`. الـTelegram bot داخل نفس Colab يستعمل هذا العنوان مباشرة. إذا ظهر `API_BASE` يمكن استعماله من خدمة خارج Colab.

In [ ]:
import requests
print(requests.get('http://127.0.0.1:8090/ping', timeout=10).status_code)
print(requests.get('http://127.0.0.1:8090/docs', timeout=10).status_code)
